In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pytorch-lightning wandb

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, shutil
!cp /content/drive/MyDrive/DL/Assignment-3/kse-ua-location-extraction-2026.zip /content/
!unzip -q -o /content/kse-ua-location-extraction-2026.zip -d /content/

drive_src, local_src = "/content/drive/MyDrive/DL/Assignment-3/src", "/content/src"
if os.path.exists(drive_src) and not os.path.exists(local_src):
    shutil.copytree(drive_src, local_src)

In [3]:
import sys, os
if '/content' not in sys.path:
    sys.path.append('/content')

import torch
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import WandbLogger
import wandb
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
import torch.nn.functional as F

from src.preprocessing import (
    hard_processing, comp_metric, clean_entity_boundaries,
    WordTokenizer, WordVocab, CharVocab, extract_entities_from_probs
)
from src.dataset import load_dataset_csv, create_dataloaders, build_vocab_from_texts
from src.models import RecurrentNERLightningModule

SEED = 42
pl.seed_everything(SEED, workers=True)

INFO:lightning_fabric.utilities.seed:Seed set to 42


42

In [4]:
class Config:
    DATA_DIR = '/content'
    CHECKPOINT_DIR = '/content/drive/MyDrive/DL/Assignment-3/checkpoints'
    SUBMISSION_DIR = '/content/drive/MyDrive/DL/Assignment-3'

    NUM_CLASSES = 7  # O, B-LOC, I-LOC, B-ORG, I-ORG, B-PER, I-PER
    MAX_LEN, MAX_CHAR_LEN = 128, 16
    BATCH_SIZE, VAL_BATCH_SIZE = 64, 128
    MAX_EPOCHS = 5

    # Emphasize LOC classes [1, 2] heavily, penalize ORG/PER [3, 4, 5, 6] to push false positives there
    CLASS_WEIGHTS = [1.0, 15.0, 15.0, 5.0, 5.0, 5.0, 5.0]

    N_SPLITS = 3  # 3-Fold Ensemble
    UA_SAMPLES, RU_SAMPLES = 300000, 100000
    WANDB_PROJECT = 'kse-ua-location-extraction-2026'

os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)

In [5]:
df_test = load_dataset_csv('test.csv', Config.DATA_DIR)
df_sample = load_dataset_csv('labeling_sample.csv', Config.DATA_DIR)
df_uk = load_dataset_csv('uk_geo_dataset.csv', Config.DATA_DIR, nrows=Config.UA_SAMPLES)
df_ru = load_dataset_csv('ru_geo_dataset.csv', Config.DATA_DIR, nrows=Config.RU_SAMPLES)

df_train_full = pd.concat([df_uk, df_ru], ignore_index=True)
print(f"Blended Dataset Size: {len(df_train_full)}")

def has_location(markers):
    s = str(markers).strip()
    return 1 if (s and s != '[]' and s != 'nan') else 0

df_train_full['has_loc'] = df_train_full['loc_markers'].apply(has_location)
skf = StratifiedKFold(n_splits=Config.N_SPLITS, shuffle=True, random_state=SEED)
df_train_full['fold'] = -1
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df_train_full, df_train_full['has_loc'])):
    df_train_full.loc[val_idx, 'fold'] = fold_idx

tokenizer = WordTokenizer(lowercase=False)
vocab = build_vocab_from_texts(df_train_full['text'].tolist(), tokenizer=tokenizer, min_freq=2, max_size=60000)
char_vocab = CharVocab()

# DataLoader for Real-World Benchmark (Labeling Sample) to tune Threshold
_, benchmark_loader, _, _ = create_dataloaders(df_sample, val_df=df_sample, vocab=vocab, char_vocab=char_vocab, tokenizer=tokenizer, batch_size=Config.BATCH_SIZE, val_batch_size=Config.VAL_BATCH_SIZE, max_len=Config.MAX_LEN)

Blended Dataset Size: 400000


In [6]:
best_model_paths = []
fold_thresholds = []

for active_fold in range(Config.N_SPLITS):
    print(f"\\n--- STARTING FOLD {active_fold} ---")
    train_df = df_train_full[df_train_full['fold'] != active_fold].reset_index(drop=True)
    val_df = df_train_full[df_train_full['fold'] == active_fold].reset_index(drop=True)

    train_loader, val_loader, test_loader, _ = create_dataloaders(
        train_df=train_df, val_df=val_df, test_df=df_test, vocab=vocab, char_vocab=char_vocab, tokenizer=tokenizer,
        batch_size=Config.BATCH_SIZE, val_batch_size=Config.VAL_BATCH_SIZE, max_len=Config.MAX_LEN
    )

    model = RecurrentNERLightningModule(
        vocab_size=len(vocab), num_classes=Config.NUM_CLASSES, embedding_dim=128, hidden_dim=256, num_layers=2,
        class_weights=Config.CLASS_WEIGHTS, max_epochs=Config.MAX_EPOCHS, use_char_cnn=True,
        char_vocab_size=len(char_vocab), char_embed_dim=32, char_feature_dim=64,
    )

    wandb_logger = WandbLogger(project=Config.WANDB_PROJECT, name=f"7class_bilstm_fold{active_fold}", log_model=False)
    checkpoint_callback = ModelCheckpoint(dirpath=Config.CHECKPOINT_DIR, filename=f"f{active_fold}_" + "{epoch:02d}_{val_comp_f1:.4f}", monitor="val_comp_f1", mode="max", save_top_k=1)

    trainer = pl.Trainer(max_epochs=Config.MAX_EPOCHS, accelerator='gpu', devices=1, logger=wandb_logger, callbacks=[checkpoint_callback], precision='16-mixed', gradient_clip_val=1.0)
    trainer.fit(model, train_loader, val_loader)

    best_model_paths.append(checkpoint_callback.best_model_path)

    # TUNE THRESHOLD ON REAL-WORLD PROXY (labeling_sample)
    best_model = RecurrentNERLightningModule.load_from_checkpoint(
    checkpoint_callback.best_model_path,
    class_weights=Config.CLASS_WEIGHTS)

    best_model.eval().cuda()

    print(f"\\nTuning Threshold on Real Data (labeling_sample) for Fold {active_fold}...")
    all_texts, all_offsets, all_gold, all_probs, all_lengths = [], [], [], [], []
    with torch.no_grad():
        for batch in benchmark_loader:
            logits = best_model(batch["input_ids"].cuda(), lengths=batch["lengths"].cuda(), char_ids=batch["char_ids"].cuda())
            probs = F.softmax(logits, dim=-1).cpu()
            all_probs.extend(list(probs))
            all_texts.extend(batch["texts"]); all_offsets.extend(batch["offsets"]); all_gold.extend(batch["gold_locations"])
            all_lengths.extend(batch["lengths"].tolist())

    best_f1, best_th = 0.0, 0.5
    for th in [0.4, 0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85]:
        y_pred = []
        for text, offsets, prob_mat, seq_len in zip(all_texts, all_offsets, all_probs, all_lengths):
            loc_probs = prob_mat[:seq_len, [1, 2]].sum(dim=-1).tolist() # SUM B-LOC + I-LOC
            y_pred.append(extract_entities_from_probs(text, offsets[:seq_len], loc_probs, threshold=th))
        th_f1 = comp_metric(all_gold, y_pred)
        if th_f1 > best_f1: best_f1, best_th = th_f1, th

    print(f"Fold {active_fold} Best Threshold: {best_th} (F1: {best_f1:.4f})")
    fold_thresholds.append(best_th)
    wandb.finish()

\n--- STARTING FOLD 0 ---


INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA L4') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
wandb: WARNING The anonymous setting has no effect and will be removed in a futur

wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ihrynkiv (ihrynkiv-assignment-3-dl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


... (7 outputs left out)

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 64. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 42. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


TypeError: RecurrentNERLightningModule.__init__() missing 1 required positional argument: 'class_weights'

In [9]:
import torch
import torch.nn.functional as F

# Якщо списки ще не створені, ініціалізуємо їх
if 'best_model_paths' not in globals():
    best_model_paths = []
if 'fold_thresholds' not in globals():
    fold_thresholds = []

# Якщо Fold 0 вже в списку, продовжуємо з Fold 1 (або запустимо решту)
start_fold = len(fold_thresholds)
print(f"Продовжуємо з фолду: {start_fold}")

for active_fold in range(start_fold, Config.N_SPLITS):
    print(f"\n--- STARTING FOLD {active_fold} ---")
    train_df = df_train_full[df_train_full['fold'] != active_fold].reset_index(drop=True)
    val_df = df_train_full[df_train_full['fold'] == active_fold].reset_index(drop=True)

    train_loader, val_loader, test_loader, _ = create_dataloaders(
        train_df=train_df, val_df=val_df, test_df=df_test, vocab=vocab, char_vocab=char_vocab, tokenizer=tokenizer,
        batch_size=Config.BATCH_SIZE, val_batch_size=Config.VAL_BATCH_SIZE, max_len=Config.MAX_LEN
    )

    model = RecurrentNERLightningModule(
        vocab_size=len(vocab), num_classes=Config.NUM_CLASSES, embedding_dim=128, hidden_dim=256, num_layers=2,
        class_weights=Config.CLASS_WEIGHTS, max_epochs=Config.MAX_EPOCHS, use_char_cnn=True,
        char_vocab_size=len(char_vocab), char_embed_dim=32, char_feature_dim=64,
    )

    wandb_logger = WandbLogger(project=Config.WANDB_PROJECT, name=f"7class_bilstm_fold{active_fold}", log_model=False)
    checkpoint_callback = ModelCheckpoint(dirpath=Config.CHECKPOINT_DIR, filename=f"f{active_fold}_" + "{epoch:02d}_{val_comp_f1:.4f}", monitor="val_comp_f1", mode="max", save_top_k=1)

    trainer = pl.Trainer(max_epochs=Config.MAX_EPOCHS, accelerator='gpu', devices=1, logger=wandb_logger, callbacks=[checkpoint_callback], precision='16-mixed', gradient_clip_val=1.0)
    trainer.fit(model, train_loader, val_loader)

    best_model_paths.append(checkpoint_callback.best_model_path)

    # TUNE THRESHOLD ON REAL-WORLD PROXY (labeling_sample) з виправленим class_weights
    best_model = RecurrentNERLightningModule.load_from_checkpoint(
        checkpoint_callback.best_model_path,
        class_weights=Config.CLASS_WEIGHTS
    )
    best_model.eval().cuda()

    print(f"\nTuning Threshold on Real Data (labeling_sample) for Fold {active_fold}...")
    all_texts, all_offsets, all_gold, all_probs, all_lengths = [], [], [], [], []
    with torch.no_grad():
        for batch in benchmark_loader:
            logits = best_model(batch["input_ids"].cuda(), lengths=batch["lengths"].cuda(), char_ids=batch["char_ids"].cuda())
            probs = F.softmax(logits, dim=-1).cpu()
            all_probs.extend(list(probs))
            all_texts.extend(batch["texts"]); all_offsets.extend(batch["offsets"]); all_gold.extend(batch["gold_locations"])
            all_lengths.extend(batch["lengths"].tolist())

    best_f1, best_th = 0.0, 0.5
    for th in [0.4, 0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85]:
        y_pred = []
        for text, offsets, prob_mat, seq_len in zip(all_texts, all_offsets, all_probs, all_lengths):
            loc_probs = prob_mat[:seq_len, [1, 2]].sum(dim=-1).tolist() # SUM B-LOC + I-LOC
            y_pred.append(extract_entities_from_probs(text, offsets[:seq_len], loc_probs, threshold=th))
        th_f1 = comp_metric(all_gold, y_pred)
        if th_f1 > best_f1: best_f1, best_th = th_f1, th

    print(f"Fold {active_fold} Best Threshold: {best_th} (F1: {best_f1:.4f})")
    fold_thresholds.append(best_th)
    wandb.finish()

print("\n Усі фолди успішно завершено!")
print("Найкращі моделі:", best_model_paths)
print("Збережені трешхолди:", fold_thresholds)

INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)


Продовжуємо з фолду: 1

--- STARTING FOLD 1 ---


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/loggers/wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ RecurrentNERNet  │ 10.2 M │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss │      0 │ train │     0 │
└───┴───────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 10.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 10.2 M                                                                                               
Total estimated model params size (MB): 40.777                                                                     
Modules in train mode: 9                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

... (14 outputs left out)

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.



Tuning Threshold on Real Data (labeling_sample) for Fold 2...
Fold 2 Best Threshold: 0.85 (F1: 0.6133)


epoch,▁▁▁▁▁▁▃▃▃▃▃▃▃▃▃▃▃▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆████
train_loss_epoch,█▄▂▂▁
train_loss_step,█▇▇▄▇▇▃▆▄▃▂▂▃▂▂▂▂▃▂▃▂▂▃▂▂▂▃▂▃▂▂▂▃▂▂▂▁▂▁▁
trainer/global_step,▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇█
val_comp_f1,▁▆▇▇█
val_loss,█▃▁▁▁
epoch,4
train_loss_epoch,0.06501
train_loss_step,0.09627
trainer/global_step,20834
val_comp_f1,0.88405



 Усі фолди успішно завершено!
Найкращі моделі: ['/content/drive/MyDrive/DL/Assignment-3/checkpoints/f0_epoch=04_val_comp_f1=0.8820.ckpt', '/content/drive/MyDrive/DL/Assignment-3/checkpoints/f0_epoch=04_val_comp_f1=0.8820.ckpt', '/content/drive/MyDrive/DL/Assignment-3/checkpoints/f1_epoch=04_val_comp_f1=0.8820.ckpt', '/content/drive/MyDrive/DL/Assignment-3/checkpoints/f2_epoch=04_val_comp_f1=0.8841.ckpt']
Збережені трешхолди: [0.85, 0.8, 0.85]


In [10]:
import os
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm

# 1. Рахуємо середній трешхолд з усіх фолдів
ensemble_th = sum(fold_thresholds) / len(fold_thresholds)
print(f"\nStarting Ensemble Inference on Test. Target Avg Threshold: {ensemble_th:.2f}")

# 2. Завантажуємо всі моделі з правильним аргументом class_weights
models = [RecurrentNERLightningModule.load_from_checkpoint(p, class_weights=Config.CLASS_WEIGHTS).eval().cuda() for p in best_model_paths]

all_text_ids, all_pred_locations = [], []
with torch.no_grad():
    for batch in test_loader:
        all_text_ids.extend(batch["text_ids"])
        input_ids, lengths, char_ids = batch["input_ids"].cuda(), batch["lengths"].cuda(), batch["char_ids"].cuda()

        # Усереднення ймовірностей (Soft Voting) по всіх фолдах
        ensemble_probs = torch.zeros(input_ids.size(0), input_ids.size(1), Config.NUM_CLASSES, device='cuda')
        for m in models:
            logits = m(input_ids, lengths=lengths, char_ids=char_ids)
            ensemble_probs += F.softmax(logits, dim=-1)
        ensemble_probs /= len(models)
        ensemble_probs = ensemble_probs.cpu() # Переносимо на CPU для постпроцесингу

        for i, text in enumerate(batch["texts"]):
            seq_len = lengths[i].item()
            loc_probs = ensemble_probs[i][:seq_len, [1, 2]].sum(dim=-1).tolist()
            ents = extract_entities_from_probs(text, batch["offsets"][i][:seq_len], loc_probs, threshold=ensemble_th)
            all_pred_locations.append(list(dict.fromkeys(ents))) # Видаляємо дублікати

# 3. Зберігаємо фінальний сабмішн
os.makedirs(Config.SUBMISSION_DIR, exist_ok=True)
submission_df = pd.DataFrame({
    'text_id': all_text_ids,
    'locations': [str(locs) for locs in all_pred_locations]
})
sub_path = os.path.join(Config.SUBMISSION_DIR, "submission_ensemble_latest.csv")
submission_df.to_csv(sub_path, index=False)

print(f"Ensemble submission saved successfully to: {sub_path}! Total rows: {len(submission_df)}")
display(submission_df.head(10))


Starting Ensemble Inference on Test. Target Avg Threshold: 0.83
Ensemble submission saved successfully to: /content/drive/MyDrive/DL/Assignment-3/submission_ensemble_latest.csv! Total rows: 477


,text_id,locations
0,0,[]
1,1,['Києві']
2,2,"['Гоголеве', 'Миргородського району Полтавсько..."
3,3,[]
4,4,['проспекті Академіка Глушкова']
5,5,['Броварах']
6,6,[]
7,7,"['Забуччя', 'Київ']"
8,8,['Дарниця Livе']
9,9,"['Солом’янському районі столиці', 'вулицею Із'..."
